# Held-out participant model comparison

This notebook evaluates PPGR predictions for participants absent from model training using five-fold grouped cross-validation. Models are fitted to free-living meals from training participants and evaluated on held-out participants.

## Notebook flow

1. Assign participants to five non-overlapping validation folds.
2. Fit the six population-level and mixed-effects model variants on training participants.
3. Generate predictions for the four outcomes in held-out participants.
4. Summarize and export performance for all validation meals and standardized meals separately.

## Outputs

CSV files written to `Results/`:

| File | Contents |
| --- | --- |
| `LOPO_CV_results.csv` | Outcome-specific R² and Pearson correlations, plus uniform-average multivariate R², by model and validation subset. |

The filename is retained for compatibility; the implemented design is five-fold participant-grouped validation.

## 1. Setup

Resolve project paths, import the modeling utilities, and load the common prepared meal-level dataset.

In [1]:
from pathlib import Path
import sys

# Find the project when launched from its root, code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
FIGURES_DIR = PROJECT_ROOT / "Figures"
RESULTS_DIR = PROJECT_ROOT / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import logging
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import xgboost as xgb
from mmer import MixedEffectEstimator
from sklearn.model_selection import BaseCrossValidator, GroupKFold
from tqdm import tqdm

In [3]:
from utils import (
    build_design_mats,
    cols_for,
    ensure_cols,
    evaluate_models,
    load_and_prepare_data,
    make_tag,
    make_xgb,
    save_performance,
    show_performance,
)

In [4]:
meta_data, data, id_to_subject_key, clusters = load_and_prepare_data(
    METADATA_PATH,
    MEAL_DATA_PATH,
)

print(f"Prepared {len(data):,} meals from {data['subject_key'].nunique():,} participants.")

data shape : (54987, 133)
Prepared 54,987 meals from 992 participants.


## 2. Participant-held-out cross-validation

In [5]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
from tqdm import tqdm
import xgboost as xgb
import statsmodels.api as sm
import logging
import statsmodels.formula.api as smf

from mmer import MixedEffectEstimator

In [7]:
logging.getLogger('merf').setLevel(logging.WARNING)

groups = clusters.to_numpy()

results_list = []

models = ["LR", "LMMER_rand_int", "LMMER_full", "XGBoost", 
          "MMER_XGBoost", 
          "MMER_XGBoost_MI"
         ]

prediction_metrics = ["max_glucose", "peak_duration", "end_glucose", "positive_iAUC"]

predictions_with_features_all_models = data.copy()

# Column-naming helpers: each (model, config) gets its own namespace ("tag")
SUFFIXES = ["true_z", "pred_z", "true", "pred", "pred_fixed_z", "pred_fixed"]

run_tags = []

participant_fold_covariances = []

######################### CV setup GROUP-K-Fold
from sklearn.model_selection import GroupKFold
cv = GroupKFold(n_splits=5)

for model_name in models:
    for std_mode in [True]:
        for interactions in [[False]]:
            for mundlak_corr in [False]:

                # Separate prediction columns for each model configuration.
                tag = make_tag(model_name, std_mode, interactions, mundlak_corr)
                cmap = cols_for(tag, prediction_metrics)
                data = ensure_cols(data, cmap, tag)
                run_tags.append(tag)

                all_y_true, all_y_pred, all_y_pred_fixed = [], [], []
                print(f"\nRunning Cross-Validation for {tag}...")

                pbar = tqdm(cv.split(data, data["net_iAUC"], groups=groups),
                            total=cv.get_n_splits(),
                            desc=f"{tag} CV", unit="fold",
                            position=1, ncols=100, dynamic_ncols=False)

                for fold, (train_idx, val_idx) in enumerate(pbar):

                    if model_name == 'MMER_XGBoost':
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, _, _= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                        )
                        y_pred_full = y_pred_fixed = None
                        xgb_fe = make_xgb()
                        model = MixedEffectEstimator(xgb_fe, max_iter=50,
                                                     slq_steps=40, tol=1e-07, n_jobs=8)

                        groups_train_mmer = clusters_train.to_numpy().reshape(-1, 1)
                        y_train_mmer = np.asarray(y_train)
                        if y_train_mmer.ndim == 1:
                            y_train_mmer = y_train_mmer.reshape(-1, 1)

                        result = model.fit(X_train, y_train_mmer, groups_train_mmer,
                                           random_slopes=([0, 1, 2, 3],))
                        
                        participant_fold_covariances.append(result.G[0]) # store covariance matrices

                        groups_val_mmer = clusters_val.to_numpy().reshape(-1, 1)
                        y_pred_full  = result.predict(X_val, groups=groups_val_mmer)
                        y_pred_fixed = result.fixed_effects_model.predict(X_val)

                    elif model_name == 'MMER_XGBoost_MI':
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, _, _= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                        )
                        y_pred_full = y_pred_fixed = None
                        xgb_fe = make_xgb()
                        model = MixedEffectEstimator(xgb_fe, max_iter=50,
                                                     slq_steps=40, tol=1e-07, n_jobs=8)

                        groups_train_mmer = clusters_train.to_numpy().reshape(-1, 1)
                        y_train_mmer = np.asarray(y_train)
                        if y_train_mmer.ndim == 1:
                            y_train_mmer = y_train_mmer.reshape(-1, 1)

                        result = model.fit(X_train, y_train_mmer, groups_train_mmer) # no random slopes

                        groups_val_mmer = clusters_val.to_numpy().reshape(-1, 1)
                        y_pred_full  = result.predict(X_val, groups=groups_val_mmer)
                        y_pred_fixed = result.fixed_effects_model.predict(X_val)

                    elif model_name == "LR":
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, train_dataframe, validation_dataframe= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                            imputation="participant_mean",
                        )
                        fe_model = LinearRegression().fit(X_train, y_train)

                        y_pred_full  = fe_model.predict(X_val)
                        y_pred_fixed = y_pred_full

                    elif model_name == "LMMER_full":
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, train_dataframe, validation_dataframe= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                            imputation="participant_mean",
                        )
                        fe_model = LinearRegression()
                        model = MixedEffectEstimator(fe_model, max_iter=50,
                                                     slq_steps=40, tol=1e-07, n_jobs=8)

                        groups_train_mmer = clusters_train.to_numpy().reshape(-1, 1)
                        y_train_mmer = np.asarray(y_train)
                        if y_train_mmer.ndim == 1:
                            y_train_mmer = y_train_mmer.reshape(-1, 1)

                        result = model.fit(X_train, y_train_mmer, groups_train_mmer,
                                           random_slopes=([0, 1, 2, 3],))

                        groups_val_mmer = clusters_val.to_numpy().reshape(-1, 1)
                        y_pred_full  = result.predict(X_val, groups=groups_val_mmer)
                        y_pred_fixed = result.fixed_effects_model.predict(X_val)

                    elif model_name == "LMMER_rand_int":
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, train_dataframe, validation_dataframe= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                            imputation="participant_mean",
                        )
                        fe_model = LinearRegression()
                        model = MixedEffectEstimator(fe_model, max_iter=50,
                                                     slq_steps=40, tol=1e-07, n_jobs=8)

                        groups_train_mmer = clusters_train.to_numpy().reshape(-1, 1)
                        y_train_mmer = np.asarray(y_train)
                        if y_train_mmer.ndim == 1:
                            y_train_mmer = y_train_mmer.reshape(-1, 1)

                        result = model.fit(X_train, y_train_mmer, groups_train_mmer,) # no random intercepts

                        groups_val_mmer = clusters_val.to_numpy().reshape(-1, 1)
                        y_pred_full  = result.predict(X_val, groups=groups_val_mmer)
                        y_pred_fixed = result.fixed_effects_model.predict(X_val)

                    elif model_name == 'XGBoost':
                        X_train, X_val, Z_train, Z_val, y_train, y_val, \
                            clusters_train, clusters_val, cluster_mapping, info_, _, _= \
                            build_design_mats(
                            data, train_idx, val_idx,
                            target_outcome=prediction_metrics,
                            apply_mundlak=mundlak_corr,
                            use_interactions=interactions,
                            population_z_scoring_features=True,
                            remove_std_meals_from_training=True,
                            remove_features=["demographics"],
                            random_slopes=["carb_eaten", "fat_eaten",
                                           "protein_eaten", "fiber_eaten"],
                            population_z_scoring_outcome=True,
                        )
                        xgb_fe = make_xgb().fit(X_train, y_train)

                        y_pred_full  = xgb_fe.predict(X_val)
                        y_pred_fixed = y_pred_full

                    # z-score representation
                    n = len(val_idx)
                    y_val_z   = np.asarray(y_val).reshape(n, -1)
                    y_pred_z  = np.asarray(y_pred_full).reshape(n, -1)
                    y_fixed_z = np.asarray(y_pred_fixed).reshape(n, -1)

                    # original-unit representation
                    if "outcome_mean" in info_:
                        y_mean = np.asarray(info_["outcome_mean"])
                        y_std  = np.asarray(info_["outcome_std"])
                        y_val_orig   = y_val_z   * y_std + y_mean
                        y_pred_orig  = y_pred_z  * y_std + y_mean
                        y_fixed_orig = y_fixed_z * y_std + y_mean
                    else:
                        y_val_orig   = y_val_z.copy()
                        y_pred_orig  = y_pred_z.copy()
                        y_fixed_orig = y_fixed_z.copy()

                    all_y_true.append(y_val_orig)
                    all_y_pred.append(y_pred_orig)
                    all_y_pred_fixed.append(y_fixed_orig)

                    # Write predictions by suffix in metric order.
                    labels = data.index[val_idx]
                    data.loc[labels, cmap["true_z"]]       = y_val_z
                    data.loc[labels, cmap["pred_z"]]       = y_pred_z
                    data.loc[labels, cmap["true"]]         = y_val_orig
                    data.loc[labels, cmap["pred"]]         = y_pred_orig
                    data.loc[labels, cmap["pred_fixed_z"]] = y_fixed_z
                    data.loc[labels, cmap["pred_fixed"]]   = y_fixed_orig
                    data.loc[labels, f"{tag}__fold"]       = fold

# Fold-level predictions remain in data for the result summaries below.


Running Cross-Validation for LR...



LR CV: 100%|████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.39fold/s]



Running Cross-Validation for LMMER_rand_int...



Converged: tolerance reached!:  40%|████      | 20/50 00:09                 | 0/5 [00:00<?, ?fold/s]

Converged: tolerance reached!:  40%|████      | 20/50 00:09             5 [00:10<00:40, 10.23s/fold]

Converged: tolerance reached!:  38%|███▊      | 19/50 00:09             5 [00:20<00:30, 10.33s/fold]

Converged: tolerance reached!:  38%|███▊      | 19/50 00:09             5 [00:30<00:20, 10.21s/fold]

Converged: tolerance reached!:  36%|███▌      | 18/50 00:08             5 [00:40<00:09,  9.94s/fold]

LMMER_rand_int CV: 100%|████████████████████████████████████████████| 5/5 [00:49<00:00,  9.88s/fold]



Running Cross-Validation for LMMER_full...


...


XGBoost CV: 100%|███████████████████████████████████████████████████| 5/5 [01:19<00:00, 15.90s/fold]



Running Cross-Validation for MMER_XGBoost...



Finished: no further improvement!:  36%|███▌      | 18/50 04:47             | 0/5 [00:00<?, ?fold/s]

Finished: no further improvement!:  50%|█████     | 25/50 06:34          [05:04<20:16, 304.23s/fold]

Finished: no further improvement!:  48%|████▊     | 24/50 06:17          [11:55<18:21, 367.31s/fold]

Finished: no further improvement!:  42%|████▏     | 21/50 06:02          [18:30<12:39, 379.80s/fold]

Finished: no further improvement!:  42%|████▏     | 21/50 05:54          [24:50<06:19, 379.75s/fold]

MMER_XGBoost CV: 100%|█████████████████████████████████████████████| 5/5 [31:00<00:00, 372.19s/fold]



Running Cross-Validation for MMER_XGBoost_MI...



Finished: no further improvement!:  18%|█▊        | 9/50 02:23              | 0/5 [00:00<?, ?fold/s]

Finished: no further improvement!:  14%|█▍        | 7/50 01:51         5 [02:39<10:39, 159.90s/fold]

Finished: no further improvement!:  14%|█▍        | 7/50 01:51         5 [04:48<07:04, 141.52s/fold]

Finished: no further improvement!:  18%|█▊        | 9/50 02:24         5 [06:56<04:30, 135.45s/fold]

Finished: no further improvement!:  18%|█▊        | 9/50 02:25         5 [09:38<02:25, 145.63s/fold]

MMER_XGBoost_MI CV: 100%|██████████████████████████████████████████| 5/5 [12:20<00:00, 148.10s/fold]


In [8]:
models = ["LR", "LMMER_rand_int", "LMMER_full", "XGBoost", 
          "MMER_XGBoost", "MMER_XGBoost_MI"
         ]
run_tags = []

######################### CV setup GROUP-K-Fold
from sklearn.model_selection import GroupKFold
cv = GroupKFold(n_splits=5)

for model_name in models:
    for std_mode in [True]:
        for interactions in [[False]]:
            for mundlak_corr in [False]:

                # Separate prediction columns for each model configuration.
                tag = make_tag(model_name, std_mode, interactions, mundlak_corr)
                run_tags.append(tag)

## 3. Performance summaries

Summarize and export the predictions from the five participant-held-out folds.

In [9]:
perf = evaluate_models(data, run_tags, prediction_metrics)
show_performance(perf, prediction_metrics)


── all meals (R² | Pearson r) ──
Model                          n   LMMER_full LMMER_rand_int           LR MMER_XGBoost MMER_XGBoost_MI      XGBoost
Metric                                                                                                             
max_glucose                54987  0.41 | 0.65    0.41 | 0.65  0.42 | 0.65  0.53 | 0.73     0.54 | 0.74  0.56 | 0.75
peak_duration              54987  0.30 | 0.56    0.30 | 0.55  0.31 | 0.56  0.41 | 0.64     0.41 | 0.64  0.43 | 0.65
end_glucose                54987  0.32 | 0.58    0.32 | 0.59  0.36 | 0.60  0.38 | 0.63     0.39 | 0.63  0.42 | 0.65
positive_iAUC              54987  0.25 | 0.50    0.24 | 0.49  0.26 | 0.52  0.39 | 0.63     0.40 | 0.63  0.42 | 0.65
Multivariate (uniform R²)  54987         0.32           0.32         0.34         0.43            0.44         0.46

── standardized (R² | Pearson r) ──
Model                         n    LMMER_full LMMER_rand_int            LR MMER_XGBoost MMER_XGBoost_MI      XGBoost


{'all meals': Model                          n   LMMER_full LMMER_rand_int           LR  \
 Metric                                                                      
 max_glucose                54987  0.41 | 0.65    0.41 | 0.65  0.42 | 0.65   
 peak_duration              54987  0.30 | 0.56    0.30 | 0.55  0.31 | 0.56   
 end_glucose                54987  0.32 | 0.58    0.32 | 0.59  0.36 | 0.60   
 positive_iAUC              54987  0.25 | 0.50    0.24 | 0.49  0.26 | 0.52   
 Multivariate (uniform R²)  54987         0.32           0.32         0.34   
 
 Model                     MMER_XGBoost MMER_XGBoost_MI      XGBoost  
 Metric                                                               
 max_glucose                0.53 | 0.73     0.54 | 0.74  0.56 | 0.75  
 peak_duration              0.41 | 0.64     0.41 | 0.64  0.43 | 0.65  
 end_glucose                0.38 | 0.63     0.39 | 0.63  0.42 | 0.65  
 positive_iAUC              0.39 | 0.63     0.40 | 0.63  0.42 | 0.65  
 Multivariate

In [10]:
metric_order = prediction_metrics + ["Multivariate (uniform R²)"]
perf = evaluate_models(data, run_tags, prediction_metrics)
save_performance(perf, metric_order, RESULTS_DIR / "LOPO_CV_results.csv")

Saved performance table: Results/LOPO_CV_results.csv


Model                                     LMMER_full LMMER_rand_int  \
all meals    max_glucose                 0.41 | 0.65    0.41 | 0.65   
             peak_duration               0.30 | 0.56    0.30 | 0.55   
             end_glucose                 0.32 | 0.58    0.32 | 0.59   
             positive_iAUC               0.25 | 0.50    0.24 | 0.49   
             Multivariate (uniform R²)          0.32           0.32   
standardized max_glucose                -0.12 | 0.46   -0.15 | 0.45   
             peak_duration              -0.32 | 0.27   -0.37 | 0.27   
             end_glucose                 0.17 | 0.47    0.17 | 0.47   
             positive_iAUC              -0.31 | 0.06   -0.34 | 0.04   
             Multivariate (uniform R²)         -0.15          -0.17   

Model                                             LR MMER_XGBoost  \
all meals    max_glucose                 0.42 | 0.65  0.53 | 0.73   
             peak_duration               0.31 | 0.56  0.41 | 0.64   
             end_glucose                 0.36 | 0.60  0.38 | 0.63   
             positive_iAUC               0.26 | 0.52  0.39 | 0.63   
             Multivariate (uniform R²)          0.34         0.43   
standardized max_glucose                -0.19 | 0.46  0.34 | 0.60   
             peak_duration              -0.32 | 0.29  0.09 | 0.35   
             end_glucose                 0.20 | 0.48  0.21 | 0.46   
             positive_iAUC              -0.38 | 0.09  0.13 | 0.36   
             Multivariate (uniform R²)         -0.17         0.19   

Model                                  MMER_XGBoost_MI      XGBoost  
all meals    max_glucose                   0.54 | 0.74  0.56 | 0.75  
             peak_duration                 0.41 | 0.64  0.43 | 0.65  
             end_glucose                   0.39 | 0.63  0.42 | 0.65  
             positive_iAUC                 0.40 | 0.63  0.42 | 0.65  
             Multivariate (uniform R²)            0.44         0.46  
standardized max_glucose                   0.34 | 0.59  0.34 | 0.59  
             peak_duration                 0.09 | 0.35  0.13 | 0.37  
             end_glucose                   0.21 | 0.47  0.22 | 0.48  
             positive_iAUC                 0.14 | 0.37  0.14 | 0.37  
             Multivariate (uniform R²)            0.19         0.21